## Init

In [1]:
import sys

sys.path.append("../src/")

In [2]:
import os
import wandb
import time
import datetime
import json
from pathlib import Path
from omegaconf import OmegaConf
import hydra
from hydra.utils import instantiate

import torch
import torch.nn as nn
import torch_geometric
import torch.backends.cudnn as cudnn
import numpy as np
from torch.cuda import amp
from torchinfo import summary

from constants import INPT_VARS, EXTRA_VARS, OUT_VARS
from utils.train_utils import loss_KE_pointwise, SmoothedValue, MetricLogger
from utils.dist_utils import (
    set_seed,
    init_distributed_mode,
    get_world_size,
    get_rank,
    is_main_process,
    all_reduce_mean,
)
from utils.data_utils import (
    data_CNN_Lateral,
    data_CNN_steps_Lateral,
    data_CNN_Dynamic,
    data_CNN_steps_Dynamic,
)


from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
import copy
from datetime import datetime
import os

## 3D Train

### 3D Surface

In [3]:
# G3D Surface
with initialize_config_dir(
    version_base=None,
    config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
):
    args = compose(
        config_name="exp/train_unet_global_3D",
        overrides=[
            "output_dir=./temp/{0}_train_convnext_3D_surface".format(str(datetime.now())[:10]),
            "nets_dir=./temp/{0}_train_convnext_3D_surface/nets_dir".format(str(datetime.now())[:10]),
            "data_dir=/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/save_data/2024-06-16-save_3D_data_surface_test/surface/data",
            "region=global_3D",
            "depth_mode=surface",
            "batch_size=8",
            "scheduler=True",
            "testing=true",
            "+name=train_convnext_3D_surface_disk"
        ],
    )
if not os.path.exists(args.output_dir):
    os.mkdir(args.output_dir)

## Setup

In [4]:

# Distributed mode
init_distributed_mode(args)
cudnn.benchmark = True

# Set seeds
set_seed(args.rand_seed)
wandb = args.wandb.mode == "online"

# Check dirs
if not os.path.exists(args.nets_dir):
    os.makedirs(args.nets_dir, exist_ok=True)

# Getting input, extra input and output
inputs = INPT_VARS[args.exp_num_in]
extra_in = EXTRA_VARS[args.exp_num_extra]
outputs = OUT_VARS[args.exp_num_out]

str_in = "".join([i + "_" for i in inputs])
str_ext = "".join([i + "_" for i in extra_in])
str_out = "".join([i + "_" for i in outputs])

print("inputs: " + str_in)
print("extra inputs: " + str_ext)
print("outputs: " + str_out)

s_train = args.lag*args.hist
e_train = s_train + args.N_samples*args.interval
e_test = e_train + args.interval*args.N_val

N_atm = len(extra_in)  # Number of atmosphere variables
N_in = len(inputs)
if args.lateral:
    N_extra = (
        N_atm + N_in
    )  # Number of atmosphere variables + Lateral boundary variables
else:
    N_extra = N_atm  # Number of atmosphere variables
N_out = len(outputs)

num_in = int((args.hist + 1) * N_in + N_extra)

print("Number of inputs: ", num_in)  # 3 (ocean speeds + ocean temp)(t) +
# 3 (atm wind stresses + atm temp)(t) +
# 3 (boundary ocean speeds + boundary ocean temp)(t) -> 3 (ocean speeds + ocean temp)(t+1)
print("Number of outputs: ", N_out)  # 3

assert args.region == "global_3D"

str_video = (
    "steps_"
    + str(args.steps)
    + "_"
    + args.region
    + "_Test_in_"
    + str_in
    + "ext_"
    + str_ext
    + "_out"
    + str_out
    + "N_train_"
    + str(args.N_samples)
    + "_Lateral_Data_025_no_smooth"
)


[17:25:57.166497] inputs: uo_vo_thetao_so_zos_
[17:25:57.166634] extra inputs: tauuo_tauvo_hfds_
[17:25:57.166673] outputs: uo_vo_thetao_so_zos_
[17:25:57.167007] Number of inputs:  8
[17:25:57.167051] Number of outputs:  5


### loaders

In [18]:
import xarray as xr
data = xr.open_zarr('/scratch/as15415/Data/Emulation_Data/OM4_Horizontal_Regrid_Old.zarr')
data['so'] = data['so'][:,0]
data['thetao'] = data['thetao'][:,0]
data['uo'] = data['uo'][:,0]
data['vo'] = data['vo'][:,0]
data_mean = data.mean()
data_std = data.std()

In [55]:
class data_CNN_Disk(torch.utils.data.Dataset):

    def __init__(self,data,inputs_str,extra_in_str,outputs_str,
                 wet,data_mean,data_std,
                 n_samples,lag,interval,ind_start,device = "cuda"):
        super().__init__()
        self.device = device        

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.ind_start = ind_start
        
        self.inputs = data[inputs_str+extra_in_str]
        self.outputs = data[outputs_str]
        
        self.in_mean = data_mean[inputs_str+extra_in_str]
        self.in_std = data_std[inputs_str+extra_in_str]
        
        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]        
        
        self.wet = wet
                    
    def set_device(self,device):
        self.device = device
    
    def __len__(self):
        # Number of data point we have. Alternatively self.data.shape[0], or self.label.shape[0]
        return self.size

    def __getitem__(self, idx):
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        ind_in = self.ind_start+idx*self.interval
        ind_out = self.ind_start+idx*self.interval + self.lag
            
        data_in = self.inputs.isel(time=ind_in)
        data_in = ((data_in-self.in_mean)/self.in_std).fillna(0)
        label = self.outputs.isel(time=ind_out)
        label = ((label-self.out_mean)/self.out_std).fillna(0)

        print(data_in.to_array().shape)
        print(label.to_array().shape)
        print(data_in.to_array().to_numpy().shape)
        # data_in = data_in.to_array().to_numpy()
        # label = label.to_array().to_numpy()            
            
        items = (torch.from_numpy(data_in).to(device = self.device),
                 torch.from_numpy(label).to(device = self.device))

        return items
    
class data_CNN_Disk_steps(torch.utils.data.Dataset):

    def __init__(self,data,inputs_str,extra_in_str,outputs_str,
                 wet,data_mean,data_std,
                 n_samples,lag,interval,steps,device = "cuda"):
        super().__init__()
        self.device = device        

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.steps = steps
        self.inputs = data[inputs_str+extra_in_str]
        self.outputs = data[outputs_str]
        
        self.in_mean = data_mean[inputs_str+extra_in_str]
        self.in_std = data_std[inputs_str+extra_in_str]
        
        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]        
        
        self.wet = wet
                    
    def set_device(self,device):
        self.device = device
    
    def __len__(self):
        # Number of data point we have. Alternatively self.data.shape[0], or self.label.shape[0]
        return self.size

    def __getitem__(self, idx):
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        outputs = []
        for step in range(self.steps):
            ind_in = idx*self.interval + self.lag*step
            ind_out = idx*self.interval + self.lag*(step+1)    
                
            data_in = self.inputs.isel(time=ind_in)
            data_in = ((data_in-self.in_mean)/self.in_std).fillna(0)
            label = self.outputs.isel(time=ind_out)
            label = ((label-self.out_mean)/self.out_std).fillna(0)
            
            if type(idx) == int:
                data_in = data_in.to_array().to_numpy()
                label = label.to_array().to_numpy()      
            else:
                data_in = data_in.to_array().to_numpy()
                label = label.to_array().to_numpy()    
                
            outputs.append(torch.from_numpy(data_in).to(device = self.device))
            outputs.append(torch.from_numpy(label).to(device = self.device))

        return outputs  

In [56]:
# Dataloaders
print("Loading data")

wet = torch.load(
    Path(args.data_dir) / "wet_data_cnn_{0}.pt".format(str_video)
)

### Normal Load
# train_data = torch.load(
#     Path(args.data_dir) / "train_data_cnn_{0}.pt".format(str_video),
#     map_location=torch.device("cpu"),
# )
# if args.data_percent < 1.0:
#     train_data = torch.utils.data.Subset(
#         train_data, list(range(int(args.N_samples * args.data_percent)))
#     )

# val_data = torch.load(
#     Path(args.data_dir) / "val_data_cnn_{0}.pt".format(str_video)
# )

### Disk Load
train_data = data_CNN_Disk_steps(data, inputs, extra_in, outputs,
                     wet, data_mean, data_std,
                     args.N_samples, args.lag, args.interval, args.steps, device="cpu")


val_data = data_CNN_Disk(data, inputs, extra_in, outputs,
                     wet, data_mean, data_std,
                     args.N_val, args.lag, args.interval, e_train, device="cpu")  

print("Instantiating torch loaders")

train_sampler = torch.utils.data.distributed.DistributedSampler(
    train_data, shuffle=True, seed=args.rand_seed
)
train_loader = torch_geometric.loader.DataLoader(
    train_data,
    batch_size=args.batch_size,
    sampler=train_sampler,
    num_workers=1,
    pin_memory=args.pin_mem,
    persistent_workers = True
)

test_sampler = torch.utils.data.DistributedSampler(
    val_data, num_replicas=get_world_size(), rank=get_rank(), shuffle=False
)
test_loader = torch_geometric.loader.DataLoader(
    val_data,
    batch_size=args.batch_size,
    sampler=test_sampler,
    num_workers=1,
    pin_memory=args.pin_mem,
    persistent_workers = True
)


[17:49:28.094210] Loading data
[17:49:28.187935] Instantiating torch loaders


In [57]:
test_loader.dataset[0]
# train_loader.dataset[0]

[17:49:29.121243] (8, 180, 360)
[17:49:29.122227] (5, 180, 360)


KeyboardInterrupt: 

### Model

In [20]:

# Model
print("Getting model " + args.network)
if "swin" == args.network:
    model = instantiate(
        args.swin,
        in_channels=num_in,
        output_channels=N_in,
        pretrain_img_size=[*train_loader.dataset[0][0].shape[1:]],
        wet=wet.cuda(),
    )
elif "convnextunet" == args.network or "adamunet" == args.network:
    model = instantiate(args.unet, wet=wet.cuda())
else:
    raise NotImplementedError

model_parameters = filter(lambda p: p.requires_grad, model.parameters())
params = sum([np.prod(p.size()) for p in model_parameters])
print("Number of parameters: ", params)
# summary(model)

model = model.to(args.device)
if args.preload:
    print("Loaded model from ", args.preload)
    model.load_state_dict(
        torch.load(args.preload, map_location=torch.device(args.device))
    )
# i = [torch.zeros(1, *train_loader.dataset[0][0].shape).cuda()] * 2
# summary(
#     model,
#     input_data=[i],
#     col_names=["kernel_size", "output_size", "num_params"],
#     depth=10,
# )

# i = [torch.zeros(1, *train_loader.dataset[0][0].shape).cuda()] * 8
# summary(model, input_data=[i], col_names=[], depth=10)

model = nn.SyncBatchNorm.convert_sync_batchnorm(model)
if "swin" in args.network:
    model = nn.parallel.DistributedDataParallel(
        model, device_ids=[args.gpu], find_unused_parameters=True
    )
elif "unet" in args.network:
    model = nn.parallel.DistributedDataParallel(model, device_ids=[args.gpu])

nets_dir = args.nets_dir
network = args.network
device = args.device

# Loss function
lam = args.lam
mse = nn.MSELoss()
if args.loss == "mse":
    print("Using mse loss")
    loss_fn = lambda out, pred: mse(out, pred)
elif args.loss == "mse_ke":
    print("lam KE: ", lam)
    loss_fn = (
        lambda out, pred: mse(out, pred) * (1 - lam)
        + loss_KE_pointwise(out, pred) * lam
    )

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
step_weights = [
    1.0
] * args.steps  # Constant weighting of losses across steps

# Scheduler
scheduler = None
if args.scheduler:
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, args.T)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, args.T
    )

# Gscaler
# gscaler = amp.GradScaler(enabled=args.grad_clip)

# Training
epochs = args.epochs
hist = args.hist
steps = args.steps
save_freq = args.save_freq
output_dir = args.output_dir
network = args.network
testing = args.testing

[17:31:21.701465] Getting model convnextunet
[17:31:21.833097] Number of parameters:  15892793
[17:31:21.860130] Using mse loss


## Train

In [11]:
def train_one_epoch(epoch):
    model.train(True)
    metric_logger = MetricLogger(delimiter="  ")
    metric_logger.add_meter("lr", SmoothedValue(window_size=1, fmt="{value:.6f}"))
    header = "Epoch: [{}]".format(epoch)
    iters = len(train_loader)

    for data_iter_step, data in enumerate(
        metric_logger.log_every(train_loader, 1, header)
    ):
        print(data_iter_step)
        if testing and (data_iter_step + 1) % 5 == 0:
            break
        optimizer.zero_grad()

        loss = model(data, loss_fn=loss_fn)
        loss.backward()
        loss_value = loss.item()

        optimizer.step()
        if scheduler is not None:
            # scheduler.step()
            scheduler.step(epoch + data_iter_step / iters)
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

        metric_logger.update(loss=loss_value)

        lr = (
            optimizer.param_groups[-1]["lr"]
            if scheduler is None
            else scheduler.get_last_lr()[0]
        )
        metric_logger.update(lr=lr)

        loss_value_reduce = all_reduce_mean(loss_value)

        if wandb:
            wandb.log(
                {
                    "epoch": epoch,
                    "train_loss_per_batch": loss_value_reduce,
                    "lr_per_batch": lr,
                }
            )

    metric_logger.synchronize_between_processes()
    print("Averaged train stats:", metric_logger)
    return {k: meter.global_avg for k, meter in metric_logger.meters.items()}

@torch.no_grad()
def validate():
    model.eval()

    metric_logger = MetricLogger(delimiter="  ")
    header = "Test:"
    for data in test_loader:
        with torch.no_grad():
            loss = model(data, loss_fn=loss_fn)
            loss_value = loss.item()
            metric_logger.update(loss=loss_value)

            loss_value_reduce = all_reduce_mean(loss_value)
            if wandb:
                wandb.log({"eval_loss_per_batch": loss_value_reduce})

    metric_logger.synchronize_between_processes()
    print("Averaged eval stats:", metric_logger)

    return {k: meter.global_avg for k, meter in metric_logger.meters.items()}


In [12]:
import datetime 

best_loss = torch.tensor(1e8)

if wandb:
    wandb.watch(model, log="all")

start_time = time.time()
for epoch in range(epochs):
    print("Epoch : {0}".format(epoch))
    start_epoch_time = time.time()
    train_sampler.set_epoch(epoch)
    test_sampler.set_epoch(epoch)

    train_stats = train_one_epoch(epoch)
    val_stats = validate()

    print("Epoch Time Elapsed: {0}".format(str(datetime.timedelta(seconds=int(time.time() - start_epoch_time)))))

    v_loss = val_stats["loss"]

    log_stats = {
        **{f"train_{k}": v for k, v in train_stats.items()},
        **{f"eval_{k}": v for k, v in val_stats.items()},
        "epoch": epoch,
    }

    if is_main_process():
        with open(
            Path(output_dir) / "log.txt", mode="a", encoding="utf-8"
        ) as f:
            f.write(json.dumps(log_stats) + "\n")

        if v_loss < best_loss:
            print("Achieved Best Validation Loss = {:5.3f}".format(v_loss))
            best_loss = v_loss
            print("Saving best model at epoch {0}".format(epoch + 1))
            torch.save(
                model.module.state_dict(),
                Path(nets_dir)
                / "{0}_best_{1}.pt".format(network, str_video),
            )

        if (epoch + 1) % save_freq == 0:
            print("Saving model at epoch {0}".format(epoch + 1))
            torch.save(
                model.module.state_dict(),
                Path(nets_dir)
                / "{0}_epoch_{1}_{2}.pt".format(
                    network, epoch + 1, str_video
                ),
            )

total_time = time.time() - start_time
total_time_str = str(datetime.timedelta(seconds=int(total_time)))
print("Training time {}".format(total_time_str))


[17:26:39.861617] Epoch : 0


KeyboardInterrupt: 